# 02 — Baseline trước MLP và kiểm định trên validation

Chạy notebook 01 trước. Notebook này chỉ đọc train/validation, không đánh giá test. Metrics được thống nhất trước huấn luyện: **MSE (USD²), RMSE/MAE (USD), R² (không đơn vị)**. Kết quả, bảng và learning curves nằm ngay trong notebook.

In [ ]:
from pathlib import Path
import sys
import hashlib
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
            if (path / 'src' / 'config.py').is_file())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
ARTIFACT_DIR = ROOT / 'notebooks' / 'artifacts'
TARGET = 'median_house_value'
SEED = 42

def file_hash(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def verify_files(manifest):
    for name, expected in manifest['files'].items():
        if file_hash(ARTIFACT_DIR / name) != expected:
            raise ValueError(f'Artifact mismatch: {name}. Rerun notebooks in order.')

from src.data.loader import load_housing_data, fetch_dataset_info
from src.data.preprocessing import split_data_8_1_1, fit_preprocessor, transform_features, save_processed_splits
from src.models.persistence import save_preprocessor, load_preprocessor
%matplotlib inline

In [ ]:
from main import set_random_seed
from src.models.baseline_models import BaselineRegressionSuite
from src.models.mlp_model import build_mlp_model, train_mlp_model, predict_mlp
from src.models.persistence import save_baseline_models, save_mlp_model, load_mlp_model
from src.evaluation.metrics import RegressionEvaluator
from src.evaluation.visualization import plot_loss_and_mae_curves

preparation_path = ARTIFACT_DIR / 'preparation_manifest.json'
preparation = json.loads(preparation_path.read_text(encoding='utf-8'))
verify_files(preparation)
if preparation['raw_sha256'] != file_hash(ROOT / 'data' / 'raw' / 'housing.csv'):
    raise ValueError('Raw dataset changed. Rerun notebook 01.')
set_random_seed(preparation['seed'])
train_frame, _ = load_housing_data(ARTIFACT_DIR / 'train.csv')
val_frame, _ = load_housing_data(ARTIFACT_DIR / 'val.csv')
X_train, y_train = train_frame.drop(columns=TARGET), train_frame[TARGET]
X_val, y_val = val_frame.drop(columns=TARGET), val_frame[TARGET]
preprocessor = load_preprocessor(ARTIFACT_DIR / 'preprocessor.joblib')
train_features = transform_features(preprocessor, X_train)
val_features = transform_features(preprocessor, X_val)
print('Đã nạp train/validation và bộ tiền xử lý; không fit lại trên validation.')


## 1. Huấn luyện ba baseline
Linear Regression tạo mốc tuyến tính; Decision Tree `max_depth=5` giới hạn độ phức tạp; Random Forest `n_estimators=100`, `max_features=1.0` giảm phương sai bằng bootstrap và averaging. Cả ba dùng cùng train và preprocessing.

In [ ]:
baselines = BaselineRegressionSuite(random_state=SEED, dt_max_depth=5, rf_n_estimators=100)
baselines.fit_all(train_features, y_train)
validation_predictions = baselines.predict_all(val_features)
baseline_summary = RegressionEvaluator.summarize_models([
    RegressionEvaluator.evaluate_predictions(y_val, values, name)
    for name, values in validation_predictions.items()])
display(baseline_summary)

## 2. Kiểm chứng feature engineering trên validation
Sau khi đã có baseline, thử **ratios + log1p** bằng một preprocessor mới chỉ fit train. Chỉ so sánh Linear Regression để minh họa tác động của features; đây không phải tìm kiếm tối ưu cho mọi mô hình. Nhánh thử nghiệm không thay thế preprocessor chuẩn và không truy cập test.

Các ratios: `rooms_per_household`, `bedrooms_ratio`, `people_per_household`. Median imputation đứng trước ratios; ratios đứng trước log; scaling đứng cuối.

In [ ]:
from sklearn.linear_model import LinearRegression
feature_preprocessor = fit_preprocessor(X_train, apply_log=True, create_interactions=True)
engineered_train = transform_features(feature_preprocessor, X_train)
engineered_val = transform_features(feature_preprocessor, X_val)
feature_model = LinearRegression().fit(engineered_train, y_train)
feature_prediction = feature_model.predict(engineered_val)
feature_comparison = pd.DataFrame([
    RegressionEvaluator.evaluate_predictions(y_val, validation_predictions['Linear Regression'], 'Linear Regression - base'),
    RegressionEvaluator.evaluate_predictions(y_val, feature_prediction, 'Linear Regression - ratios + log')])
display(feature_comparison)
change = feature_comparison.iloc[1]['RMSE'] - feature_comparison.iloc[0]['RMSE']
print(f'Chênh lệch RMSE validation của nhánh thêm đặc trưng so với nhánh chuẩn: {change:,.2f} USD.')
print('Giữ cấu hình chuẩn đã chốt cho so sánh bốn mô hình; kết quả thử nghiệm đặc trưng chỉ mang tính đối chiếu.')


## MLP PyTorch và quy trình huấn luyện

Kiến trúc `32 ReLU → 16 ReLU → 1 linear`, Adam 0.001, 100 epochs, batch 32. Chuẩn hóa target bằng mean/std chỉ từ train; optimizer tối thiểu hóa MSE trên target chuẩn hóa. History và prediction trả về USD (MSE: USD²). Chạy đủ epochs rồi khôi phục checkpoint có validation MSE thấp nhất, tương đương RMSE thấp nhất; không early stopping, không clipping.


In [ ]:
EPOCHS = 100
BATCH_SIZE = 32
LEARNING_RATE = 0.001
# Đặt lại seed để nhánh thử nghiệm đặc trưng không làm đổi khởi tạo MLP.
set_random_seed(SEED)
model = build_mlp_model(train_features.shape[1], learning_rate=LEARNING_RATE)
print(model.network)
model, history = train_mlp_model(model, train_features, y_train, val_features, y_val,
                                  epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=1)


In [ ]:
history_frame = pd.DataFrame(history, index=np.arange(1, EPOCHS + 1))
history_frame.index.name = 'Epoch'
display(history_frame.tail(10))
display(plot_loss_and_mae_curves(history))
validation_predictions['MLP PyTorch'] = predict_mlp(model, val_features)
validation_summary = RegressionEvaluator.summarize_models([
    RegressionEvaluator.evaluate_predictions(y_val, values, name)
    for name, values in validation_predictions.items()])
train_predictions = baselines.predict_all(train_features)
train_predictions['MLP PyTorch'] = predict_mlp(model, train_features)
train_summary = RegressionEvaluator.summarize_models([
    RegressionEvaluator.evaluate_predictions(y_train, values, name)
    for name, values in train_predictions.items()])
gap = train_summary[['Model', 'RMSE']].merge(validation_summary[['Model', 'RMSE']],
                                          on='Model', suffixes=('_Train', '_Validation'))
gap['RMSE_Gap_USD'] = gap['RMSE_Validation'] - gap['RMSE_Train']
display(validation_summary)
display(gap)
best = validation_summary.sort_values('RMSE').iloc[0]
print(f"Mô hình có RMSE validation thấp nhất trong cấu hình đã thử: {best['Model']} ({best['RMSE']:,.2f} USD).")
print(f"MSE validation MLP đầu/cuối: {history['val_loss'][0]:,.0f} / {history['val_loss'][-1]:,.0f} USD².")
print('Khoảng cách train–validation lớn là dấu hiệu cần xem xét overfitting; chưa dùng test để kết luận hoặc chỉnh tham số.')


## 4. Lưu artifact bàn giao và kiểm tra reload
Mô hình được ghép với đúng preprocessing và raw splits thông qua manifest. Lịch sử training là dữ liệu bàn giao để notebook 03 vẽ lại khi cần, không phải báo cáo xuất ngoài notebook.

In [ ]:
save_baseline_models(baselines.models, ARTIFACT_DIR / 'baseline_models.joblib')
save_mlp_model(model, ARTIFACT_DIR / 'mlp_house_price_model.pt')
(ARTIFACT_DIR / 'training_history.json').write_text(json.dumps(history), encoding='utf-8')
reloaded_model = load_mlp_model(ARTIFACT_DIR / 'mlp_house_price_model.pt')
np.testing.assert_allclose(predict_mlp(model, val_features[:3]),
                           predict_mlp(reloaded_model, val_features[:3]), rtol=1e-5, atol=1e-3)
training_manifest = {'preparation_sha256': file_hash(preparation_path),
                     'parameters': {'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE},
                     'files': {name: file_hash(ARTIFACT_DIR / name) for name in
                               ['baseline_models.joblib', 'mlp_house_price_model.pt', 'training_history.json']}}
(ARTIFACT_DIR / 'training_manifest.json').write_text(json.dumps(training_manifest, indent=2), encoding='utf-8')
print('Đã lưu mô hình cùng liên kết preprocessing; dự đoán sau nạp lại khớp trong sai số cho phép.')


## Kết luận trước khi mở test
Cấu hình bốn mô hình đã được cố định. MLP có thể hội tụ chậm với target USD và không được giả định tốt hơn Random Forest trên dữ liệu dạng bảng. Notebook 03 chỉ đánh giá, không huấn luyện lại hoặc điều chỉnh theo test.